# 07 Semantic Vector Retrieval

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `07-semantic-vector-retrieval.ipynb`

In [1]:
# Start coding here
# ==================================================
# Notebook 07
# Semantic Vector Retrieval
# ==================================================

import pickle

import faiss
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
with open("artifacts/router_embeddings.pkl", "rb") as file:

    department_embeddings = pickle.load(file)

In [4]:
def route_query(query):

    query_embedding = embedding_model.encode(query)

    scores = {}

    for department, dept_embedding in department_embeddings.items():

        score = cosine_similarity(
            query_embedding.reshape(1, -1), dept_embedding.reshape(1, -1)
        )[0][0]

        scores[department] = float(score)

    best_department = max(scores, key=scores.get)

    return best_department, scores

In [5]:
children_df = pd.read_csv("artifacts/child_chunks.csv")

children_df.head()

,child_id,parent_id,document_id,content
0,5f2b6fb4-0d03-4817-900a-945162a178c2,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,AI Platform Roadmap\nSearch Infrastructure\nRe...
1,047bf5df-d450-4bbe-92c2-2396ed3a7b30,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,ructure rollout.\nSearch latency improved sign...
2,b400d323-ca7c-4813-ac4e-a2ba69786f3f,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,eval implementation.\nSemantic search quality ...
3,e68fd4e0-8796-416e-b24b-46ccae0f8d1d,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,er deployment planned.\nEnterprise RAG platfor...
4,e001567a-7134-4290-9ebb-e7e07e069dcb,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,ent | Status\nVector Search | Production\nHybr...


In [6]:
parents_df = pd.read_csv("artifacts/parent_chunks.csv")

parents_df.head()

,parent_id,document_id,content
0,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,AI Platform Roadmap\nSearch Infrastructure\nRe...
1,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,er deployment planned.\nEnterprise RAG platfor...
2,9cb41e2d-b979-4c89-a091-01db42c3748e,b638264b-b788-442e-a09d-19176466fba8,Annual Financial Report 2026\nRevenue Performa...
3,7cdee370-fba0-4bc7-9fe1-a39230162073,b638264b-b788-442e-a09d-19176466fba8,oud division grew by 35%.\nAI products generat...
4,fd7bb5e1-9ec6-470a-8726-bbdc04fcc6dc,31516a79-2cf5-4080-ac66-66b338d6ebff,Employee Benefits 2026\nHealth Insurance\nDent...


In [7]:
departments = ["hr", "finance", "engineering"]

vector_indexes = {}

for department in departments:

    index = faiss.read_index(f"artifacts/faiss/{department}.index")

    vector_indexes[department] = index

print("Indexes loaded.")

Indexes loaded.


In [8]:
with open("artifacts/faiss/chunk_mapping.pkl", "rb") as file:

    chunk_mappings = pickle.load(file)

In [9]:
chunk_mappings.keys()

dict_keys(['engineering', 'finance', 'hr'])

In [10]:
def semantic_search(query, department, top_k=5):

    query_embedding = embedding_model.encode([query])

    query_embedding = np.array(query_embedding).astype("float32")

    faiss.normalize_L2(query_embedding)

    index = vector_indexes[department]

    scores, ids = index.search(query_embedding, top_k)

    mapping = chunk_mappings[department]

    results = []

    for score, idx in zip(scores[0], ids[0]):

        row = mapping.iloc[idx]

        results.append(
            {
                "score": float(score),
                "child_id": row["child_id"],
                "parent_id": row["parent_id"],
                "content": row["content"],
            }
        )

    return results

In [11]:
semantic_search("How much revenue was generated?", "finance")

[{'score': 0.5479419231414795,
  'child_id': 'fae64b23-bac1-4b0c-8f50-029329796b70',
  'parent_id': '9cb41e2d-b979-4c89-a091-01db42c3748e',
  'content': 'illion.\nRevenue increased across all regions.\nSubscription services contributed strongly.\nOperating '},
 {'score': 0.5404300093650818,
  'child_id': '0b338cc8-85bc-440a-b1a2-a88b6a8b035b',
  'parent_id': '9cb41e2d-b979-4c89-a091-01db42c3748e',
  'content': 'Annual Financial Report 2026\nRevenue Performance\nProfitability\nCloud Division\nRevenue reached $5.2 b'},
 {'score': 0.46930187940597534,
  'child_id': '0dcc44dc-097f-463f-b5d4-7e1614ce3ef3',
  'parent_id': '7cdee370-fba0-4bc7-9fe1-a39230162073',
  'content': '\nRegion | Revenue\nUS | 2.1B\nEurope | 1.7B\nAsia | 1.4B\n'},
 {'score': 0.45197391510009766,
  'child_id': 'f9c01c39-fe13-43cb-9297-81f5de952cca',
  'parent_id': '7cdee370-fba0-4bc7-9fe1-a39230162073',
  'content': 'oud division grew by 35%.\nAI products generated record profits.\nEnterprise customers expanded usage.'}

In [12]:
semantic_search("Explain vector search infrastructure", "engineering")

[{'score': 0.6067765951156616,
  'child_id': '5f2b6fb4-0d03-4817-900a-945162a178c2',
  'parent_id': '7d09d047-2858-43c4-bf8b-d40519129087',
  'content': 'AI Platform Roadmap\nSearch Infrastructure\nRetrieval Systems\nDeployment Roadmap\nVector search infrast'},
 {'score': 0.500562310218811,
  'child_id': 'e001567a-7134-4290-9ebb-e7e07e069dcb',
  'parent_id': 'a1c5bb38-4de7-4ca4-805a-7e3fc0419867',
  'content': 'ent | Status\nVector Search | Production\nHybrid Retrieval | Testing\nCross Encoder | Planned\n'},
 {'score': 0.42805445194244385,
  'child_id': '047bf5df-d450-4bbe-92c2-2396ed3a7b30',
  'parent_id': '7d09d047-2858-43c4-bf8b-d40519129087',
  'content': 'ructure rollout.\nSearch latency improved significantly.\nNew indexing pipeline deployed.\nHybrid retri'},
 {'score': 0.3412363827228546,
  'child_id': 'b400d323-ca7c-4813-ac4e-a2ba69786f3f',
  'parent_id': '7d09d047-2858-43c4-bf8b-d40519129087',
  'content': 'eval implementation.\nSemantic search quality improved.\nKnowledge bas

In [13]:
def get_parent_context(parent_id):

    parent = parents_df[parents_df["parent_id"] == parent_id]

    if len(parent) == 0:

        return None

    return parent.iloc[0]["content"]

In [14]:
results = semantic_search("company revenue", "finance", top_k=1)

parent_id = results[0]["parent_id"]

get_parent_context(parent_id)

'Annual Financial Report 2026\nRevenue Performance\nProfitability\nCloud Division\nRevenue reached $5.2 billion.\nRevenue increased across all regions.\nSubscription services contributed strongly.\nOperating margin improved to 18%.\nCost optimization reduced expenses.\nFree cash flow reached record levels.\nCl'

In [15]:
def semantic_retrieve(query, department, top_k=5):

    child_results = semantic_search(query, department, top_k)

    final_results = []

    for result in child_results:

        parent_context = get_parent_context(result["parent_id"])

        final_results.append(
            {
                "score": result["score"],
                "child_text": result["content"],
                "parent_context": parent_context,
            }
        )

    return final_results

In [16]:
semantic_retrieve("How much money did we generate?", "finance")

[{'score': 0.3653598725795746,
  'child_text': 'margin improved to 18%.\nCost optimization reduced expenses.\nFree cash flow reached record levels.\nCl',
  'parent_context': 'Annual Financial Report 2026\nRevenue Performance\nProfitability\nCloud Division\nRevenue reached $5.2 billion.\nRevenue increased across all regions.\nSubscription services contributed strongly.\nOperating margin improved to 18%.\nCost optimization reduced expenses.\nFree cash flow reached record levels.\nCl'},
 {'score': 0.35906684398651123,
  'child_text': 'Annual Financial Report 2026\nRevenue Performance\nProfitability\nCloud Division\nRevenue reached $5.2 b',
  'parent_context': 'Annual Financial Report 2026\nRevenue Performance\nProfitability\nCloud Division\nRevenue reached $5.2 billion.\nRevenue increased across all regions.\nSubscription services contributed strongly.\nOperating margin improved to 18%.\nCost optimization reduced expenses.\nFree cash flow reached record levels.\nCl'},
 {'score': 0.3544931

In [17]:
def routed_semantic_search(query):

    department, scores = route_query(query)

    results = semantic_retrieve(query, department)

    return {"route": department, "router_scores": scores, "results": results}

In [18]:
routed_semantic_search("What AI infrastructure projects are planned?")

{'route': 'engineering',
 'router_scores': {'hr': 0.1434076428413391,
  'finance': 0.21328476071357727,
  'engineering': 0.5762346982955933},
 'results': [{'score': 0.5611736178398132,
   'child_text': 'AI Platform Roadmap\nSearch Infrastructure\nRetrieval Systems\nDeployment Roadmap\nVector search infrast',
   'parent_context': 'AI Platform Roadmap\nSearch Infrastructure\nRetrieval Systems\nDeployment Roadmap\nVector search infrastructure rollout.\nSearch latency improved significantly.\nNew indexing pipeline deployed.\nHybrid retrieval implementation.\nSemantic search quality improved.\nKnowledge base coverage expanded.\nCross encod'},
  {'score': 0.3374946117401123,
   'child_text': 'er deployment planned.\nEnterprise RAG platform launch Q3.\nAgentic workflows under evaluation.\nCompon',
   'parent_context': 'er deployment planned.\nEnterprise RAG platform launch Q3.\nAgentic workflows under evaluation.\nComponent | Status\nVector Search | Production\nHybrid Retrieval | Testing\nCro

In [19]:
retrieval_config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "index_type": "FAISS IndexFlatIP",
    "retrieval": "semantic",
}

import json

with open("artifacts/semantic_config.json", "w") as file:

    json.dump(retrieval_config, file, indent=4)